# Community Detection

### Import libraries

In [1]:
import numpy as np
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt
import seaborn as sns
import time
import dimod
from networkx.algorithms.community.quality import modularity as nx_modularity
from dwave.system import LeapHybridSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.system import FixedEmbeddingComposite
from dwave.system import LeapHybridDQMSampler
from dwave.samplers import SimulatedAnnealingSampler
from dwave.embedding import embed_bqm
from dwave.embedding.chain_strength import uniform_torque_compensation
from minorminer import find_embedding
from dimod import ExactSolver

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

### Load data

In [3]:
G = nx.read_graphml('neurons5.graphml')
G.remove_nodes_from(list(nx.isolates(G)))
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph loaded: 124 nodes, 246 edges


# 1 Graph preprocessing

### 1.2 Conversion to adjacency matrix

In [ ]:
# Convert directed graph to undirected
G_undirected = G.to_undirected()

# Remove any remaining self-loops
G_undirected.remove_edges_from(nx.selfloop_edges(G_undirected))

# Create unweighted adjacency matrix
A = nx.adjacency_matrix(G_undirected, weight=None).toarray()
N = G_undirected.number_of_nodes()
M = G_undirected.number_of_edges()  # Total number of edges

print(f"\nUndirected graph: {N} nodes, {M} edges")
print(f"Adjacency matrix shape: {A.shape}")
print(f"Sparsity: {(A == 0).sum() / A.size * 100:.2f}% zeros")
print(f"\nDegree statistics:")
degrees = [G_undirected.degree(n) for n in G_undirected.nodes()]
print(f"  Min: {min(degrees)}, Max: {max(degrees)}, Mean: {np.mean(degrees):.2f}")


Undirected graph: 124 nodes, 123 edges
Adjacency matrix shape: (124, 124)
Sparsity: 98.40% zeros

Degree statistics:
  Min: 1, Max: 6, Mean: 1.98


# 2 Classical community detection

In [13]:
def communities_to_partition(communities):
    """
    Convert a list of community sets to a partition array.
    Returns: array where partition[i] = community_id for node i
    """
    node_list = list(G_undirected.nodes())  # Use the current graph's node order
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}
    partition = np.zeros(len(node_list), dtype=int)
    for community_id, community_set in enumerate(communities):
        for node in community_set:
            partition[node_to_idx[node]] = community_id
    return partition

def compute_modularity(partition, A):
    """
    Compute modularity Q for a given partition.
    Q = (1/2m) * sum_ij (A_ij - k_i*k_j/2m) * delta(c_i, c_j)
    where k_i is the degree of node i, and delta(c_i, c_j) = 1 if nodes i,j in same community
    """
    n = A.shape[0]

    degrees = A.sum(axis=1)
    m = A.sum() / 2.0  # total edges (double counted in adjacency matrix)
    
    Q = 0.0
    for i in range(n):
        for j in range(n):
            if partition[i] == partition[j]:
                Q += A[i, j] - (degrees[i] * degrees[j]) / (2.0 * m)
    
    Q /= (2.0 * m)
    return Q

def compute_conductance(partition, A):
    """
    Compute conductance for a given partition.
    Conductance = cut(S, S') / min(vol(S), vol(S'))
    where cut(S, S') is the number of edges between S and its complement,
    and vol(S) is the sum of degrees in S.
    """
    n = A.shape[0]

    degrees = A.sum(axis=1)
    conductances = []
    for community_id in np.unique(partition):
        S = np.where(partition == community_id)[0]

        # Skip single-node communities (undefined conductance)
        if len(S) <= 1:
            continue

        S_complement = np.where(partition != community_id)[0]
        cut = A[np.ix_(S, S_complement)].sum()
        vol_S = degrees[S].sum()
        vol_S_complement = degrees[S_complement].sum()

        if min(vol_S, vol_S_complement) > 0:
            conductance = cut / min(vol_S, vol_S_complement)
            conductances.append(conductance)

    return np.mean(conductances) if conductances else 0.0

### 2.1 Louvain

In [17]:
start_time = time.time()
communities_louvain = list(nx.community.louvain_communities(G_undirected, seed=42))
louvain_runtime = time.time() - start_time

partition_louvain = communities_to_partition(communities_louvain)
modularity_louvain = nx_modularity(G_undirected, communities_louvain, weight=None)
# conductance_louvain = compute_conductance(partition_louvain, A)

print(f"Runtime: {louvain_runtime:.4f}s")
print(f"Number of communities: {len(communities_louvain)}")
print(f"Modularity: {modularity_louvain:.6f}")
# print(f"Conductance: {conductance_louvain:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_louvain], reverse=True)}")
print(f"Number of communities: {len(communities_louvain)}")

Runtime: 0.0222s
Number of communities: 55
Modularity: 0.530240
Community sizes: [11, 8, 8, 7, 6, 5, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Number of communities: 55


### 2.2 Greedy modularity optimization

In [18]:
start_time = time.time()
communities_greedy = list(nx.community.greedy_modularity_communities(G_undirected))
greedy_runtime = time.time() - start_time

partition_greedy = communities_to_partition(communities_greedy)
modularity_greedy = nx_modularity(G_undirected, communities_greedy, weight=None)
# conductance_greedy = compute_conductance(partition_greedy, A)

print(f"Runtime: {greedy_runtime:.4f}s")
print(f"Number of communities: {len(communities_greedy)}")
print(f"Modularity: {modularity_greedy:.6f}")
# print(f"Conductance: {conductance_greedy:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_greedy], reverse=True)}")

Runtime: 0.0212s
Number of communities: 21
Modularity: 0.829929
Community sizes: [12, 10, 10, 10, 9, 9, 9, 8, 8, 7, 5, 5, 4, 3, 3, 2, 2, 2, 2, 2, 2]


In [19]:
import math
start_time = time.time()
communities_greedy = list(nx.community.greedy_modularity_communities(G_undirected, best_n = math.sqrt(124)))
greedy_runtime = time.time() - start_time

partition_greedy = communities_to_partition(communities_greedy)
modularity_greedy = nx_modularity(G_undirected, communities_greedy, weight=None)
# conductance_greedy = compute_conductance(partition_greedy, A)

print(f"Runtime: {greedy_runtime:.4f}s")
print(f"Number of communities: {len(communities_greedy)}")
print(f"Modularity: {modularity_greedy:.6f}")
# print(f"Conductance: {conductance_greedy:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_greedy], reverse=True)}")

Runtime: 0.0288s
Number of communities: 11
Modularity: 0.254743
Community sizes: [97, 5, 4, 3, 3, 2, 2, 2, 2, 2, 2]


# 3 QUBO with Modularity

### 3.0 Build QUBO matrix

In [20]:
def build_modularity_qubo(A, num_communities=None, lambda_penalty=None, verbose=False):
    """
    Build QUBO matrix for modularity maximization via binary community assignment.
    Variables: x_{i,c} for i in nodes, c in communities
    Constraint: Each node assigned to exactly one community (handled via penalty)
    
    One-hot encoding:
    - num_binary_vars = N * num_communities
    - x[i*num_communities + c] = 1 if node i in community c
    
    QUBO objective: minimize -Q + lambda * penalty
    
    Parameters:
    -----------
    A : ndarray
        Adjacency matrix
    num_communities : int, optional
        Number of communities (default: sqrt(N))
    lambda_penalty : float, optional
        Penalty weight for one-hot constraint.
        If None, automatically set to 10x the max modularity contribution.
    verbose : bool
        Print diagnostics
    """

    if num_communities is None:
        num_communities = max(2, int(np.sqrt(N)))

    n_vars = N * num_communities 
    Q = {}

    degrees = A.sum(axis=1)
    m = A.sum() / 2  # Total edges

    # Modularity term: Q = (1/2m) * sum_{c} sum_{i,j} B_{ij} x_{i,c} x_{j,c}
    # For minimization, use -Q in QUBO objective
    max_B = 0  # Track max modularity contribution for scaling
    
    for c in range(num_communities):
        for i in range(N):
            for j in range(i, N):
                var_i = i * num_communities + c
                var_j = j * num_communities + c
                B_ij = A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
                coeff = -B_ij / (2 * m)
                max_B = max(max_B, abs(coeff))

                if i == j:  # diagonal
                    if var_i not in Q:
                        Q[var_i] = 0.0
                    Q[var_i] += coeff
                else:  # off-diagonal
                    key = (var_i, var_j) if var_i < var_j else (var_j, var_i)
                    if key not in Q:
                        Q[key] = 0.0
                    Q[key] += coeff

    if lambda_penalty is None:
        lambda_penalty = max(10.0 * max_B, 5.0)  # At least 5.0
    
    if verbose:
        print(f"Modularity scale (max |coeff|): {max_B:.6f}")
        print(f"Penalty weight (lambda): {lambda_penalty:.6f}")

    # Constraint penalty: each node in exactly one community
    # Penalty: lambda * (sum_c x_{i,c} - 1)^2
    # Expansion:
    #   (sum_c x_{i,c} - 1)^2 = sum_c x_{i,c}^2 + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} - 2*sum_c x_{i,c} + 1
    #   = sum_c x_{i,c} + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} - 2*sum_c x_{i,c} + 1   [since x^2=x for binary]
    #   = -sum_c x_{i,c} + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} + 1
    
    # Linear coefficient: -lambda per variable
    # Quadratic coefficient: +2*lambda per pair

    const_term = N * lambda_penalty
    
    for i in range(N):
        # Linear terms: -lambda * x_{i,c}
        for c in range(num_communities):
            var_i = i * num_communities + c
            if var_i not in Q:
                Q[var_i] = 0.0
            Q[var_i] += -lambda_penalty

        # Quadratic terms: +2*lambda * x_{i,c1}*x_{i,c2}
        for c1 in range(num_communities):
            for c2 in range(c1 + 1, num_communities):
                var_1 = i * num_communities + c1
                var_2 = i * num_communities + c2
                key = (var_1, var_2) if var_1 < var_2 else (var_2, var_1)
                if key not in Q:
                    Q[key] = 0.0
                Q[key] += 2.0 * lambda_penalty

    Q['__const__'] = const_term
    
    if verbose:
        print(f"Constant term (N*lambda): {const_term:.2f}")
        print(f"Total QUBO terms: {len(Q)-1} + 1 const")  # -1 for const term itself

    return Q, num_communities

In [21]:
# Estimate reasonable number of communities
num_communities_target = max(2, int(np.sqrt(N)))
print(f"Target number of communities: {num_communities_target}")
print(f"Total QUBO variables: {N * num_communities_target}")

# Build QUBO
print(f"\nBuilding QUBO matrix...")
Q_dict, k_communities = build_modularity_qubo(A, num_communities_target, verbose=True)
print(f"QUBO built with {len(Q_dict)-1} terms")
print(f"Constant term: {Q_dict.get('__const__', 0):.2f}")

Target number of communities: 11
Total QUBO variables: 1364

Building QUBO matrix...
Modularity scale (max |coeff|): 1.287787
Penalty weight (lambda): 12.877874
Constant term (N*lambda): 1596.86
Total QUBO terms: 92070 + 1 const
QUBO built with 92070 terms
Constant term: 1596.86


In [22]:
# Convert QUBO to BQM (Binary Quadratic Model) for D-Wave
Q_dict_clean = {}
const_offset = 0

for k, v in Q_dict.items():
    if k == '__const__':
        # Extract constant term for later addition
        const_offset = float(v)
    elif isinstance(k, tuple):
        Q_dict_clean[k] = float(v)
    else:
        node_idx = int(k)
        Q_dict_clean[(node_idx, node_idx)] = float(v)

# Create BQM and add the constant offset
bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict_clean)

# Add constant term to the BQM's offset
bqm.offset += const_offset

print(f"BQM created: {bqm.num_variables} variables, {len(bqm.quadratic)} interactions")
print(f"BQM offset (constant term): {bqm.offset:.2f}")

BQM created: 1364 variables, 90706 interactions
BQM offset (constant term): 1596.86


### 3.1 Classical solvers

Simulated annealing

In [23]:
# Simplified A LOT because 100 reads takes too long
start_time = time.time()
sa_sampler = SimulatedAnnealingSampler()
sampleset_sa = sa_sampler.sample(bqm, num_reads=100, seed=42)
sa_runtime = time.time() - start_time

best_sample_sa = sampleset_sa.first.sample
best_energy_sa = sampleset_sa.first.energy

print(f"Runtime: {sa_runtime:.4f}s")
print(f"Best energy: {best_energy_sa:.6f}")
print(f"Number of reads: 100")
print(f"Top 5 solutions:")

# Convert generator into list
top5 = list(sampleset_sa.data(['sample', 'energy', 'num_occurrences'], sorted_by='energy'))[:5]
for i, (sample, energy, num_occurrences) in enumerate(top5):
    print(f"  {i+1}. Energy: {energy:.6f}, Occurrences: {num_occurrences}")

Runtime: 8.0640s
Best energy: 2.955405
Number of reads: 100
Top 5 solutions:
  1. Energy: 2.955405, Occurrences: 1
  2. Energy: 3.975055, Occurrences: 1
  3. Energy: 3.994041, Occurrences: 1
  4. Energy: 4.014989, Occurrences: 1
  5. Energy: 4.071772, Occurrences: 1


In [26]:
def extract_communities_from_sample(sample, N, k_communities):
    """
    Extract community assignments from QUBO sample.
    For node i, find c where x_{i,c} = 1
    """
    partition = np.zeros(N, dtype=int)
    
    for i in range(N):
        # Find which community this node is assigned to
        for c in range(k_communities):
            var_idx = i * k_communities + c
            if var_idx in sample and sample[var_idx] == 1:
                partition[i] = c
                break
        else:
            # If no community assigned (constraint violation), assign to community 0
            partition[i] = 0
    
    return partition
count = 1

for sample, energy, num_occurrences in top5:
    partition_sa = extract_communities_from_sample(sample, N, k_communities)
    modularity_sa = nx_modularity(G_undirected, [set(np.where(partition_sa == c)[0]) for c in np.unique(partition_sa)], weight=None)
    num_communities_found_sa = len(np.unique(partition_sa))
    
    print(f"\nPartition from sample with energy {energy:.6f}:")
    print(f"\nSample {count}:")
    print(f"  Number of communities found: {num_communities_found_sa}")
    print(f"  Modularity: {modularity_sa:.6f}")
    print(f"  Community sizes: {sorted(np.bincount(partition_sa), reverse=True)}")
    count += 1

NotAPartition: [{np.int64(3), np.int64(6), np.int64(70), np.int64(40), np.int64(9), np.int64(11), np.int64(13), np.int64(45), np.int64(46), np.int64(110), np.int64(81), np.int64(56), np.int64(58), np.int64(59), np.int64(61)}, {np.int64(36), np.int64(7), np.int64(73), np.int64(106), np.int64(15), np.int64(19), np.int64(52), np.int64(92)}, {np.int64(100), np.int64(105), np.int64(76), np.int64(109), np.int64(14), np.int64(17), np.int64(85), np.int64(118), np.int64(90), np.int64(29)}, {np.int64(75), np.int64(44), np.int64(77), np.int64(80), np.int64(113), np.int64(117), np.int64(91)}, {np.int64(97), np.int64(68), np.int64(78), np.int64(95)}, {np.int64(1), np.int64(101), np.int64(71), np.int64(47), np.int64(84), np.int64(23), np.int64(63)}, {np.int64(64), np.int64(33), np.int64(4), np.int64(37), np.int64(8), np.int64(104), np.int64(10), np.int64(42), np.int64(12), np.int64(107), np.int64(49), np.int64(18)}, {np.int64(32), np.int64(96), np.int64(2), np.int64(35), np.int64(72), np.int64(43), 

### 3.2 Quantum/hybrid solvers

In [28]:
# Before using D-Wave hybrid solve, check feature availability
# Dsampler = DWaveSampler(solver={'name': 'Advantage_system4.1'})
Dsampler = DWaveSampler(solver={'name': 'Advantage2_system1'})
print("Connected to sampler", Dsampler.solver.name)

Connected to sampler Advantage2_system1


In [29]:
def greedy_refinement(A, partition):
    """
    Refines the partition using a greedy local search.
    Moves each node to the community that gives the highest modularity increase.
    """
    refined_partition = partition.copy()
    N = A.shape[0]
    unique_communities = np.unique(partition)
    improved = True
    
    current_modularity = compute_modularity(refined_partition, A)
    
    print(f"Starting refinement... Initial Modularity: {current_modularity:.6f}")
    
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        
        for i in nodes:
            best_move = refined_partition[i]
            initial_comm = refined_partition[i]
            for target_comm in unique_communities:
                if target_comm == initial_comm:
                    continue
                refined_partition[i] = target_comm
                new_modularity = compute_modularity(refined_partition, A)
                if new_modularity > current_modularity:
                    current_modularity = new_modularity
                    best_move = target_comm
                    improved = True
            refined_partition[i] = best_move
            
    print(f"Refinement complete. Optimized Modularity: {current_modularity:.6f}")
    return refined_partition

def advanced_refinement(A, partition):
    N = A.shape[0]
    m = np.sum(A) / 2
    degrees = np.sum(A, axis=1)
    refined_partition = partition.copy()
    
    def get_delta_q(node_i, target_comm, curr_partition):
        # Calculate only the change in modularity for moving node_i
        # Internal edges to the target community
        comm_nodes = np.where(curr_partition == target_comm)[0]
        ki_in = np.sum(A[node_i, comm_nodes])
        ki = degrees[node_i]
        sum_tot = np.sum(degrees[comm_nodes])
        
        # Simplified Delta Q formula
        return (ki_in / m) - (sum_tot * ki) / (2 * m**2)

    improved = True
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        for i in nodes:
            old_comm = refined_partition[i]
            # Only check communities that node i is actually connected to (saves time!)
            neighbor_nodes = np.where(A[i, :] > 0)[0]
            neighbor_comms = np.unique(refined_partition[neighbor_nodes])
            
            best_delta = 0
            best_move = old_comm
            
            for target_comm in neighbor_comms:
                if target_comm == old_comm: continue
                
                delta_q = get_delta_q(i, target_comm, refined_partition)
                if delta_q > best_delta:
                    best_delta = delta_q
                    best_move = target_comm
                    improved = True
            
            refined_partition[i] = best_move
            
    return refined_partition

3.2a D-Wave LeapHybrid (quantum+classical, requires Leap account)

In [30]:
start_time = time.time()
sampler = LeapHybridSampler()
sampleset_hybrid = sampler.sample(bqm, time_limit=10)

# target_edgelist = sampler.edgelist
# find_embedding(bqm.quadratic, target_edgelist)

# sampleset_hybrid = hybrid_sampler.sample(bqm, time_limit=1)
hybrid_runtime = time.time() - start_time

best_sample_hybrid = sampleset_hybrid.first.sample
best_energy_hybrid = sampleset_hybrid.first.energy
# use_hybrid = True

In [33]:
# Refine with greedy optimization
partition_hybrid_raw = extract_communities_from_sample(best_sample_hybrid, N, k_communities)
print("Applying greedy refinement to hybrid solution...")
partition_hybrid_optimized = greedy_refinement(A, partition_hybrid_raw)
partition_hybrid = extract_communities_from_sample(best_sample_hybrid, N, k_communities)

num_communities_found_hybrid = len(np.unique(partition_hybrid))
# conductance_hybrid = compute_conductance(partition_hybrid, A)
mod_raw = nx_modularity(G_undirected, set([np.where(partition_hybrid_raw == c)[0] for c in np.unique(partition_hybrid_raw)]), weight=None)
mod_opt = nx_modularity(G_undirected, set([np.where(partition_hybrid_optimized == c)[0] for c in np.unique(partition_hybrid_optimized)]), weight=None)

print(f"Runtime: {hybrid_runtime:.4f}s")
print(f"Best energy: {best_energy_hybrid:.6f}")
print(f"Number of communities found: {num_communities_found_hybrid}")
print(f"Original Hybrid Modularity: {mod_raw:.6f}")
print(f"Optimized Hybrid Modularity: {mod_opt:.6f}")
# print(f"Conductance: {conductance_hybrid:.6f}")
print(f"Community sizes: {sorted(np.bincount(partition_hybrid), reverse=True)}")

Applying greedy refinement to hybrid solution...
Starting refinement... Initial Modularity: 2.691058
Refinement complete. Optimized Modularity: 14.107913


TypeError: unhashable type: 'numpy.ndarray'

3.2b D-Wave QPU sampler (requires D-Wave account)

In [ ]:
import time
import numpy as np

qpu_sampler = EmbeddingComposite(Dsampler)
total_actual_qpu_micros = 0

# parameters
variables = list(bqm.variables)
chunk_size = 100
current_sample = {}
for i in range(N):
    # Randomly assign each node to ONE community (one-hot)
    comm = np.random.randint(0, k_communities)
    for c in range(k_communities):
        current_sample[i * k_communities + c] = 1 if c == comm else 0
total_qpu_time = 0

print(f"Starting QPU decomposition loop on: {Dsampler.properties['chip_id']}")

for i in range(0, len(variables), chunk_size):
    # Select a subset of variables to optimize
    subset = variables[i : i + chunk_size]

    fixed_vars = {v: current_sample[v] for v in variables if v not in subset}
    sub_bqm = bqm.copy()
    sub_bqm.fix_variables(fixed_vars)
    
    loop_start = time.time()
    
    try:
        sampleset = qpu_sampler.sample(sub_bqm, num_reads=1000, chain_strength=2.0, annealing_time=100, label=f'Part {i}')
        qpu_micros = sampleset.info['timing']['qpu_access_time']
        total_actual_qpu_micros += qpu_micros
        best_sub_sample = sampleset.first.sample
        current_sample.update(best_sub_sample)
        
    except ValueError as e:
        print(f"Sub-problem {i} failed: {e}")
        continue
        
    loop_end = time.time()
    
    elapsed = loop_end - loop_start
    total_qpu_time += elapsed
    
    print(f"Chunk {i//chunk_size + 1}: processed in {elapsed:.4f}s")

print(f"\nTotal Loop Runtime: {total_qpu_time:.4f}s")
total_qpu_seconds = total_actual_qpu_micros / 1_000_000
print(f"Total Pure QPU Access Time: {total_qpu_seconds:.6f}s")
best_sample_qpu = current_sample
partition_qpu = np.zeros(N, dtype=int)

Starting QPU decomposition loop on: Advantage_system4.1
Chunk 1: processed in 139.0347s
Chunk 2: processed in 107.2441s
Chunk 3: processed in 103.5052s
Chunk 4: processed in 119.7448s
Chunk 5: processed in 186.1488s
Chunk 6: processed in 76.6210s

Total Loop Runtime: 732.2986s
Total Pure QPU Access Time: 1.701283s


In [19]:
# Postprocessing
partition_qpu = np.zeros(N, dtype=int)
unassigned_nodes = 0

for i in range(N):
    node_vars = [current_sample.get(i * k_communities + c, 0) for c in range(k_communities)]
    
    if sum(node_vars) == 1:
        partition_qpu[i] = np.argmax(node_vars)
    elif sum(node_vars) > 1:
        partition_qpu[i] = np.argmax(node_vars)
    else:
        unassigned_nodes += 1
        partition_qpu[i] = 0 # This defaults to Community 0
print(f"Nodes failed by QPU: {unassigned_nodes} out of {N}")

print("Applying advanced refinement to QPU solution...")
partition_qpu_optimized = advanced_refinement(A, partition_qpu)

modularity_qpu = compute_modularity(partition_qpu_optimized, A)
conductance_qpu = compute_conductance(partition_qpu_optimized, A)
print(f"Advanced Optimized QPU Modularity: {modularity_qpu:.6f}")
print(f"Advanced Optimized QPU Conductance: {conductance_qpu:.6f}")
print(f"Community sizes: {sorted(np.bincount(partition_qpu_optimized), reverse=True)}")

Nodes failed by QPU: 0 out of 70
Applying advanced refinement to QPU solution...
Advanced Optimized QPU Modularity: 0.308915
Advanced Optimized QPU Conductance: 0.088031
Community sizes: [np.int64(41), np.int64(26), np.int64(3), np.int64(0), np.int64(0), np.int64(0)]


3.2c Hybrid

In [ ]:
# Variant A: Classical vs Quantum racing
workflow_v1 = hybrid.Loop(
    hybrid.RacingBranches(
        hybrid.InterruptableTabuSampler(),
        hybrid.QPUSubproblemAutoEmbeddingSampler()
    ) | hybrid.ArgMin(),
    convergence=3)

# Variant B: Multiple classical heuristics racing
workflow_v2 = hybrid.Loop(
    hybrid.RacingBranches(
        hybrid.InterruptableTabuSampler(),
        hybrid.SteepestDescentSolver(),
        hybrid.RandomSampler()
    ) | hybrid.ArgMin(),
    convergence=3)

# Variant C: Decompose then race
workflow_v3 = hybrid.Loop(
    hybrid.EnergyImpactDecomposer(size=40, rolling=True)
    | hybrid.RacingBranches(
        hybrid.QPUSubproblemAutoEmbeddingSampler(),
        hybrid.GreedyPathMergeSampler()
    ) | hybrid.ArgMin(),
    convergence=3)

In [24]:
import hybrid
# Set a workflow of tabu search in parallel to submissions to a D-Wave system
workflow = hybrid.Loop(
   hybrid.RacingBranches(
      hybrid.InterruptableTabuSampler(),
      hybrid.EnergyImpactDecomposer(size=30, rolling=True, rolling_history=0.75)
      | hybrid.QPUSubproblemAutoEmbeddingSampler()
      | hybrid.SplatComposer()) | hybrid.ArgMin(), convergence=3)

# Convert to dimod sampler and run workflow
result = hybrid.HybridSampler(workflow).sample(bqm)

In [40]:
best_sample_hybrid = result.first.sample
best_energy_hybrid = result.first.energy

# Refine with greedy optimization
partition_hybrid_raw = extract_communities_from_sample(result, N, k_communities)
print("Applying greedy refinement to hybrid solution...")
partition_hybrid_optimized = greedy_refinement(A, partition_hybrid_raw)
partition_hybrid = extract_communities_from_sample(result, N, k_communities)

num_communities_found_hybrid = len(np.unique(partition_hybrid))
conductance_hybrid = compute_conductance(partition_hybrid, A)
mod_raw = compute_modularity(partition_hybrid_raw, A)
mod_opt = compute_modularity(partition_hybrid_optimized, A)

print(f"Best energy: {best_energy_hybrid:.6f}")
print(f"Number of communities found: {num_communities_found_hybrid}")
print(f"Original Hybrid Modularity: {mod_raw:.6f}")
print(f"Optimized Hybrid Modularity: {mod_opt:.6f}")
print(f"Conductance: {conductance_hybrid:.6f}")
print(f"Community sizes: {sorted(np.bincount(partition_hybrid), reverse=True)}")

Applying greedy refinement to hybrid solution...
Starting refinement... Initial Modularity: -0.000000
Refinement complete. Optimized Modularity: -0.000000
Graph contains 0 single-node communities
Best energy: -700.153106
Number of communities found: 1
Original Hybrid Modularity: -0.000000
Optimized Hybrid Modularity: -0.000000
Conductance: 0.000000
Community sizes: [np.int64(70)]


# 4 Discrete Quadratic Model

In [41]:
def build_modularity_dqm(A, num_communities):
    """
    Build DQM (Discrete Quadratic Model) for modularity maximization.

    Variables: x_i for i in nodes, where x_i in {0, 1, ..., num_communities-1}
    x_i = c means node i is in community c

    Objective: minimize -Q, where Q is modularity
    """
    from dimod import DiscreteQuadraticModel
    
    dqm = DiscreteQuadraticModel()
    
    # Add variables: each node has discrete variable with num_communities possible values
    for i in range(N):
        dqm.add_variable(num_communities, label=i)
    
    degrees = A.sum(axis=1)
    m = A.sum() / 2
    
    # Modularity term: Q = (1/2m) * sum_{i<j} B_ij * delta(x_i, x_j)
    # Since DQM minimizes, we use -Q as objective
    for i in range(N):
        for j in range(i+1, N):
            B_ij = A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
            coeff = -B_ij / (2 * m)  # Negative because we minimize -Q
            
            # For each community c, add quadratic term when both nodes are in c
            for c in range(num_communities):
                dqm.set_quadratic(i, j, {(c, c): coeff})
    
    return dqm


In [48]:
# Build DQM
print(f"\nBuilding DQM matrix...")
dqm = build_modularity_dqm(A, num_communities_target)
print(f"DQM built with {dqm.num_variables()} variables")



Building DQM matrix...
DQM built with 70 variables


In [46]:
start_time = time.time()
dqm_sampler = LeapHybridDQMSampler()
sampleset_dqm = dqm_sampler.sample_dqm(dqm, time_limit=10)
dqm_runtime = time.time() - start_time

best_sample_dqm = sampleset_dqm.first.sample
best_energy_dqm = sampleset_dqm.first.energy

print(f"DQM Runtime: {dqm_runtime:.4f}s")
print(f"Best energy: {best_energy_dqm:.6f}")


DQM Runtime: 9.8574s
Best energy: -0.195486


In [47]:
# Extract communities from DQM sample
partition_dqm = np.array([best_sample_dqm[i] for i in range(N)])
modularity_dqm = compute_modularity(partition_dqm, A)
conductance_dqm = compute_conductance(partition_dqm, A)
num_communities_found_dqm = len(np.unique(partition_dqm))

print(f"Number of communities found: {num_communities_found_dqm}")
print(f"DQM Modularity: {modularity_dqm:.6f}")
print(f"DQM Conductance: {conductance_dqm:.6f}")
print(f"Community sizes: {sorted(np.bincount(partition_dqm), reverse=True)}")


Graph contains 0 single-node communities
Number of communities found: 7
DQM Modularity: 0.350623
DQM Conductance: 0.213847
Community sizes: [np.int64(33), np.int64(10), np.int64(10), np.int64(7), np.int64(4), np.int64(3), np.int64(3), np.int64(0)]


### Visualizations

### Comparison